# Course 3 · Week 2 — Hands-on: Collaborative Filtering + PCA

By the end you'll have:

1. Implemented the collaborative-filtering cost function
2. Implemented its gradients (with respect to BOTH user vectors and item vectors)
3. Trained a CF model on a 5-user × 6-movie matrix with 40% missing ratings
4. Predicted the missing ratings and verified MAE on truth
5. Computed PCA via SVD on a 2-D dataset
6. (Stretch) Reconstructed data from the first principal component

Estimated time: **45–60 minutes.**


## Setup — the rating matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 5 users × 6 movies. Underlying truth: each movie has a 2-d "genre vector"
# (romance score, action score), each user has a 2-d "taste vector".
# Predicted rating = user_taste · movie_genre. Then we hide ~40% of ratings.

np.random.seed(0)
movie_features = np.array([
    [1.0, 0.0],  # m0: pure romance
    [0.9, 0.1],  # m1: mostly romance
    [0.2, 0.8],  # m2: mostly action
    [0.0, 1.0],  # m3: pure action
    [0.5, 0.5],  # m4: balanced
    [0.7, 0.3],  # m5: mostly romance
])
user_features = np.array([
    [4.0, 0.5],
    [3.5, 1.0],
    [0.5, 4.0],
    [1.0, 3.5],
    [2.0, 2.0],
])
R_full = np.clip(user_features @ movie_features.T, 0, 5)

m_users, m_items = R_full.shape
np.random.seed(42)
mask = np.random.rand(m_users, m_items) < 0.6
R_obs = np.where(mask, R_full, np.nan)
print("Observed ratings (NaN = not rated):")
print(np.round(R_obs, 2))
print(f"\n{int(mask.sum())} ratings observed, {int((~mask).sum())} hidden")
R = np.nan_to_num(R_obs, nan=0.0)  # numeric form (zeros where unrated)


## Recap — collaborative filtering

The matrix `R` is partly observed: some users rated some movies, but most cells are blank. The goal is to predict the blanks.

The trick: assume each user has a `k`-dimensional taste vector `w_u`, each movie a `k`-dimensional feature vector `x_m`, and the rating is `w_u · x_m + b_m`. Both `w` and `x` are unknown — we learn them jointly by minimizing squared error over *observed* entries (plus regularization).

How does this even work? Multiple users rate the same movies. So if Alice and Bob both rate Movie 1, that gives the optimizer two equations involving Movie 1's vector — enough to start pinning it down. Once Movie 1's vector is roughly known, it constrains the user vectors of everyone who rated it. The two sides bootstrap each other.

Cost (squared error over rated entries + L2 regularization):

$$J = \frac{1}{2} \sum_{(u, m) \,:\, r_{um} \text{ observed}} (w_u \cdot x_m + b_m - r_{um})^2 + \frac{\lambda}{2}(\|W\|^2 + \|X\|^2)$$


## Exercise 1 — CF cost

In [ ]:
def cf_cost(W, X, b, R, mask, lam):
    """Collaborative-filtering cost.

    W: (n_users, k)   user vectors
    X: (n_items, k)   item vectors
    b: (n_items,)     item biases
    R: (n_users, n_items) observed ratings (zeros for unrated)
    mask: same shape, 1 if rated, 0 if not
    lam: regularization strength

    Cost: 1/2 * sum over rated entries of (W @ X.T + b - R)^2
          plus (lam/2) * (||W||^2 + ||X||^2)   — no reg on b
    """
    pred = W @ X.T + b           # (n_users, n_items)
    # TODO: compute err only on rated entries (multiply by mask), then squared sum / 2
    # TODO: add lam/2 regularization on W and X (sum of squares)
    return 0.0


# Sanity check on a tiny example
W_t = np.array([[1.0, 0.0]]); X_t = np.array([[1.0, 0.0], [0.0, 1.0]])
b_t = np.zeros(2); R_t = np.array([[1.0, 0.0]]); mask_t = np.array([[1.0, 1.0]])
# Predicted = [[1, 0]], errors = [0, 0], cost = 0
J = cf_cost(W_t, X_t, b_t, R_t, mask_t, lam=0.0)
print(f"perfect-fit cost = {J:.4f}   expect 0.0")
assert abs(J) < 1e-9
# With lam=1: reg = 0.5 * (||W||^2 + ||X||^2) = 0.5 * (1 + 2) = 1.5
J = cf_cost(W_t, X_t, b_t, R_t, mask_t, lam=1.0)
print(f"with lam=1, cost = {J:.4f}   expect 1.5")
assert abs(J - 1.5) < 1e-9
print("✓ cf_cost() works")


## Exercise 2 — CF gradients

In [ ]:
def cf_gradients(W, X, b, R, mask, lam):
    """Gradients of the CF cost."""
    pred = W @ X.T + b
    err = (pred - R) * mask
    # TODO:
    # dW = err @ X + lam * W       (n_users, k)
    # dX = err.T @ W + lam * X     (n_items, k)
    # db = err.sum(axis=0)         (n_items,)
    return np.zeros_like(W), np.zeros_like(X), np.zeros_like(b)


## Exercise 3 — train via gradient descent

In [ ]:
n_features = 2
np.random.seed(0)
W = np.random.randn(m_users, n_features) * 0.5
X = np.random.randn(m_items, n_features) * 0.5
b = np.zeros(m_items)
alpha = 0.05
lam = 0.1

print(f"Initial cost: {cf_cost(W, X, b, R, mask, lam):.4f}")

hist = []
for step in range(1000):
    # TODO: gradients, simultaneous update, log cost
    pass

print(f"Final cost: {hist[-1] if hist else 'n/a'}")

R_pred = W @ X.T + b
hidden = ~mask
mae = float(np.abs(R_pred - R_full)[hidden].mean())
print(f"\nMAE on the {int(hidden.sum())} hidden ratings: {mae:.4f}")
print(f"Predictions vs truth on hidden entries:")
for i, j in zip(*np.where(hidden)):
    print(f"  user {i}, movie {j}:  predicted {R_pred[i,j]:.2f},  true {R_full[i,j]:.2f}")


## Recap — PCA

PCA finds the direction(s) along which data varies most. The first principal component (PC1) is the unit vector along the longest axis of the data cloud.

Computationally, PCA = SVD on centered data:

$$X = U S V^T$$

The columns of `V` (rows of `Vt` in numpy) are the principal directions. The singular values `S[k]` tell you how much variance is along the k-th direction. `S[k]^2 / sum(S^2)` is the "fraction of variance explained" by that component — you'll see this metric everywhere.

PCA today is mostly used for **visualization** (squash high-dim data to 2D for plotting). Its older use case as a compression / pre-processing step has mostly been replaced by deep learning.


## Exercise 4 — PCA via SVD

In [ ]:
# PCA on a 2-D dataset that lives almost entirely along one direction.
np.random.seed(1)
n = 60
theta = np.deg2rad(30)
T = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
X_pca = np.random.randn(n, 2) * np.array([2.0, 0.3]) @ T.T
X_pca -= X_pca.mean(axis=0)

# TODO: do PCA via SVD
# 1. U, S, Vt = np.linalg.svd(X_pca, full_matrices=False)
# 2. The principal components are the rows of Vt
# 3. Variance explained by component k = S[k]^2 / sum(S^2)

U, S, Vt = (None, None, None)
# Replace the line above with the actual SVD call

if S is not None:
    print(f"singular values: {np.round(S, 3)}")
    print(f"explained variance ratio: {np.round(S**2 / (S**2).sum(), 3)}")

    # Plot original points + the first principal direction
    plt.figure(figsize=(6, 6))
    plt.scatter(X_pca[:, 0], X_pca[:, 1], color="#4338ca", s=40, alpha=0.7)
    arrow = Vt[0] * S[0] / np.sqrt(n)
    plt.arrow(0, 0, arrow[0], arrow[1], head_width=0.1, color="#ef4444", lw=2)
    plt.axis("equal"); plt.grid(alpha=0.3)
    plt.title("Data + first principal component (red arrow)")
    plt.show()


## ⭐ Stretch — reconstruct from 1 PC

In [ ]:
# STRETCH: reconstruct from 1 PC, plot reconstruction error
# 1. Project X_pca onto first PC: z = X_pca @ Vt[0]
# 2. Reconstruct: X_recon = np.outer(z, Vt[0])  (back to 2D)
# 3. MSE = mean((X_pca - X_recon)^2)
# Verify the reconstruction is close to the original.
